In [46]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


In [53]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGroq(model= "llama-3.3-70b-versatile")
# llm = ChatGroq(model= "qwen/qwen3-32b")
llm = ChatGroq(model= "openai/gpt-oss-120b")

# 1. Stufff Document Chain

A **stuff documents** chain in LangChain is a summarization or processing approach where all the provided documents (or chunks of text) are **directly concatenated into a single large string** and then inserted into a prompt for the LLM to process in one go. This method is simple and efficient when dealing with a small number of documents or short text chunks, since the model sees the entire context at once. However, **it can quickly hit token limits if the documents are too large**, which is why alternative strategies like **map-reduce** or refine are often preferred for bigger datasets.

[Doc1] + [Doc2] + [Doc3] + ... + [DocN]
->
[Prompt]
->
[LLM]
->
Final Summary

In [55]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("D:\DATA SCIENCE ML AI\Git\git-cheat-sheet-education.pdf")
docs = loader.load()
docs

Ignoring wrong pointing object 11 0 (offset 0)


[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': 'D:\\DATA SCIENCE ML AI\\Git\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all local repositories\ngit config --global user.name “[firs

In [56]:
from langchain_core.prompts import PromptTemplate

stuff_prompt = PromptTemplate.from_template(
  """
    Write a concise summary of the following document.\n\n
    Document: {context}
  """
)

In [57]:
from langchain.chains.combine_documents import create_stuff_documents_chain

stuff_chain = create_stuff_documents_chain(llm, stuff_prompt)

result = stuff_chain.invoke({"context": docs})

In [58]:
print(result)

**Git Cheat Sheet – Quick Summary**

- **Staging & Snapshots** – Use `git status`, `git add`, `git reset`, `git diff` (unstaged) and `git diff --staged` (staged) to view and manage changes; commit with `git commit -m "msg"`.

- **User Setup** – Configure identity globally:  
  `git config --global user.name "Name"`  
  `git config --global user.email "email"`  
  Enable colored output: `git config --global color.ui auto`.

- **Repository Init & Clone** – Create a repo in the current folder with `git init`; copy an existing one with `git clone <url>`.

- **Branching & Merging** – List/create branches with `git branch` / `git branch <name>`; switch with `git checkout`; merge with `git merge <branch>`; view history via `git log`.

- **Installation & GUIs** – Git is available for all platforms (Windows, macOS, Linux/Solaris) via installers and GUIs from GitHub or the official site.

- **Remote Collaboration** – Add a remote: `git remote add <alias> <url>`; fetch with `git fetch <alias>`; m

## Method 2 for Stuff:

In [59]:
from langchain.chains.summarize import load_summarize_chain

stuff_prompt = PromptTemplate.from_template(
  """
    Write a concise summary of the following document.\n\n
    Document: {text}
  """
)

stuff_chain = load_summarize_chain(
  llm= llm,
  chain_type= "stuff",
  verbose= True
)

stuff_chain.invoke(
  {"input_documents" : docs}
)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now to your next commit (stage)
git reset [file]
unstage a ﬁle while retaining the changes in working directory
git diff
diﬀ of what is changed but not staged
git diff --staged
diﬀ of what is staged but not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit snapshot
SETUP
Conﬁguring user information used across all local repositories
git config --global user.name “[firstname lastname]”
set a name that is identiﬁable for credit when review version history
git config --global user.email “[valid-email]”
set an email address that will be associated with each history marker
git config --global color.ui aut

{'input_documents': [Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': 'D:\\DATA SCIENCE ML AI\\Git\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all local repositories\ngit config --glo

# 2. Map Reduce Summarization

**Map-Reduce summarization** in LangChain is a **two-step summarization approach** designed for handling many or large documents without hitting token limits:

- Map step → Each document (or chunk) is **summarized individually using the LLM**.

- Reduce step → **The individual summaries are then combined**, and the LLM produces **a final, concise overall summary**.
- There can be different prompt templates for chunk summary and combined summary.

This way, the LLM never has to process the entire raw text at once, making it scalable for long PDFs, research papers, or large collections of documents.

### **MAP**
#### [Doc1] → [LLM] → [Summary1]
##### [Doc2] → [LLM] → [Summary2]
##### [Doc3] → [LLM] → [Summary3]
##### ...
##### [DocN] → [LLM] → [SummaryN]

------------------------------------------------
### **REDUCE**
##### [Summary1 + Summary2 + ... + SummaryN]
#####   ↓
##### [LLM]
#####   ↓
##### Final Summary

In [19]:
docs = PyPDFLoader("attention_is_all_you_need.pdf").load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogl

In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(docs)
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogl

In [21]:
map_prompt = PromptTemplate.from_template(
    "Write a concise summary of the following chunk of text:\n\n{text}"
)
reduce_prompt = PromptTemplate.from_template(
    "Combine the following summaries into a coherent overall summary:\n\n{text}"
)

In [22]:
from langchain.chains.summarize import load_summarize_chain
map_reduce_chain = load_summarize_chain(
    llm= llm,
    chain_type= "map_reduce",
    map_prompt= map_prompt,        # applied to each chunk
    combine_prompt= reduce_prompt, # combines partial summaries
    verbose= True,
    return_intermediate_steps= True
)

In [23]:
result = map_reduce_chain.invoke(docs)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following chunk of text:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
me

In [33]:
print(result)

{'input_documents': [Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁ

In [34]:
print(result.keys())

dict_keys(['input_documents', 'intermediate_steps', 'output_text'])


In [38]:
print("**Intermediate Steps**: \n", result["intermediate_steps"])
print("**Output**: \n", result["output_text"])

**Intermediate Steps**: 
 ['The authors, primarily from Google, propose a new neural network architecture called the Transformer, which relies solely on attention mechanisms, eliminating the need for recurrent or convolutional neural networks.', 'The Transformer, a new network architecture based on attention mechanisms, outperforms existing models in machine translation tasks, achieving state-of-the-art results in English-to-German and English-to-French translations, while requiring less training time and being more parallelizable.', 'The text discusses the development of the Transformer model, which generalizes well to various tasks, including English constituency parsing. It also acknowledges the contributions of several researchers who worked on the model, including its design, implementation, and evaluation.', 'Lukasz and Aidan worked on tensor2tensor, a project that improved research efficiency and results, and presented their work at the 2017 NIPS conference.', 'Here is a concise

# 3. Refine Chain Summarization

The Refine Chain in LangChain is an advanced method for summarizing large documents by iteratively improving a summary as more content is processed. Unlike the Stuff Chain, which attempts to summarize all content at once, or the Map-Reduce Chain, which summarizes individual chunks and then combines them in a single step, the Refine Chain maintains context throughout the summarization process. **It begins by creating an initial summary of the first chunk of text. Then, for each subsequent chunk, it takes the existing summary along with the new chunk and asks the language model to refine or enhance the summary, integrating new information while keeping the prior context intact. This iterative approach ensures that the final summary is coherent, detailed, and contextually consistent, making it particularly useful for long documents or complex content.** Although it may require more computational resources and API calls due to its step-by-step refinement, the Refine Chain produces summaries that are often more accurate and readable than those generated by the simpler Stuff or Map-Reduce approaches.

In [43]:
from langchain.chains.summarize import load_summarize_chain

initial_prompt = PromptTemplate.from_template(
    "Write a concise summary of the following document:\n\n{text}"
)

refine_prompt = PromptTemplate.from_template(
    "The existing summary is:\n{existing_answer}\n\n"
    "Refine the summary with the following document:\n{text}"
)

In [49]:
refine_chain = load_summarize_chain(
    llm= llm,
    chain_type= "refine",
    verbose= True,
    question_prompt= initial_prompt,  # used for the first chunk
    refine_prompt= refine_prompt      # used for subsequent chunks
)

In [50]:
result = refine_chain.invoke(docs)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following document:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism.

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01j8pgfewpezxa83ww17myzjbk` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 8479, Requested 3550. Please try again in 1m0.297s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}